# Week 4 — Automatic Evaluation Metrics
**DSA 4020A Semester Project — PSA Machine Translation**

Computes **SacreBLEU**, **chrF**, and **COMET** for both fine-tuned models against their proper
held-out test splits (`data/intermediate/test.csv` for English-Kiswahili, `data/intermediate/test_guz.csv`
for English-Ekegusii — neither file was used during training).

**Requires a GPU runtime and the fine-tuned model checkpoints.** If you're running this right after
`colab_training.ipynb` in the *same* Colab session, the checkpoints already exist at
`models/psa-en-sw-finetuned` and `models/nllb-en-guz`. If the session was restarted, you'll need to
re-train first (model weights aren't committed to the repo — see `.gitignore`) or load them from
wherever you persisted them (Drive, HF Hub, etc.).

In [ ]:
import os

if "COLAB_GPU" in os.environ or os.environ.get("COLAB_RELEASE_TAG"):
    if not os.path.exists("Multilogual_transaltion_nlp"):
        os.system("git clone https://github.com/aykahsay/Multilogual_transaltion_nlp.git")
    os.chdir("Multilogual_transaltion_nlp")
    os.system("pip install -q -r requirements.txt")

import sys
sys.path.insert(0, os.path.join("src", "training_eval"))

import torch
print("CUDA available:", torch.cuda.is_available())

## Direction 1: English -> Kiswahili (MarianMT)

In [ ]:
from evaluate import evaluate_model

sw_result = evaluate_model(
    model_path="models/psa-en-sw-finetuned",
    test_data_path=None,   # defaults to data/intermediate/test.csv
    target_lang="Kiswahili",
)

## Direction 2: English -> Ekegusii (NLLB-200)

In [ ]:
guz_result = evaluate_model(
    model_path="models/nllb-en-guz",
    test_data_path=None,   # defaults to data/intermediate/test_guz.csv
    target_lang="Ekegusii",
)

## Combined Results
All runs are appended to `reports/week4_automatic_metrics.csv` (one row per run, timestamped) so repeated evaluations across sessions/checkpoints stay comparable.

In [ ]:
import pandas as pd

results_path = os.path.join("reports", "week4_automatic_metrics.csv")
if os.path.exists(results_path):
    results_df = pd.read_csv(results_path)
    display(results_df)
else:
    print("No results saved yet -- did the cells above run successfully?")

### Notes on interpreting these scores
- **SacreBLEU / chrF** are surface n-gram overlap metrics — sensitive to exact wording, penalize
  valid paraphrases.
- **COMET** is a neural, source-aware metric (trained on human quality judgments) — generally
  correlates better with human fluency/adequacy ratings, especially for morphologically rich
  languages like Ekegusii where n-gram overlap metrics under-score valid translations.
- These automatic scores are **not a substitute** for the native-speaker human evaluation
  (fluency/adequacy/cultural accuracy) still required by the Week 4 checklist — treat them as a
  first-pass sanity check, not the final quality verdict.